# Build a Kokoro voice from your recording (KVoiceWalk, Kaggle)

Throwaway spike for the LekturaC Kokoro tester. Makes `dev.bin`, a Kokoro voicepack that sounds like the voice in your recording.

**Before you start (right-hand panel, Session options):**
- Accelerator: **GPU T4 x2** (not P100: the PyTorch KVoiceWalk installs no longer supports that card)
- Internet: **On** (needs a phone-verified Kaggle account)
- Input: **Add Input → Upload** your WAV as a dataset (any name)

Your recording is stored in that Kaggle dataset (private unless you make it public).

## 1. Check the GPU and find your recording

In [ ]:
import glob, os, torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'No GPU. Session options > Accelerator > GPU T4 x2, then run this cell again.'
audio_files = sorted(p for ext in ('wav', 'mp3', 'm4a', 'webm', 'ogg', 'flac')
                     for p in glob.glob(f'/kaggle/input/**/*.{ext}', recursive=True))
print('Recordings found:', audio_files)
assert audio_files, 'No recording found. Add Input > Upload > pick your WAV, then run this cell again.'

## 2. Install KVoiceWalk (about 3 to 5 minutes)
Kokoro needs Python 3.12 or older, so this makes a separate Python 3.12 just for KVoiceWalk (with `uv`).

In [ ]:
%cd /tmp
!apt-get -qq install -y espeak-ng > /dev/null
!rm -rf /tmp/kvoicewalk && git clone -q --depth 1 https://github.com/RobViren/kvoicewalk.git /tmp/kvoicewalk
assert os.path.exists('/tmp/kvoicewalk/main.py'), 'Download of KVoiceWalk failed. Is Internet on in Session options?'
!pip install -q uv librosa soundfile
%cd /tmp/kvoicewalk
!uv sync --python 3.12 --quiet
!.venv/bin/python -c "import torch, kokoro; print('KVoiceWalk Python ready | GPU:', torch.cuda.is_available())"
assert os.path.exists('/tmp/kvoicewalk/.venv/bin/python'), 'Install failed. Paste the red text above.'
%cd /kaggle/working
print('Installed.')

## 3. Convert your recording to 24 kHz mono

In [ ]:
import librosa, soundfile as sf
os.makedirs('/tmp/kvoicewalk/in', exist_ok=True)
print('Using', audio_files[0])
audio, _ = librosa.load(audio_files[0], sr=24000, mono=True)
sf.write('/tmp/kvoicewalk/in/dev.wav', audio, 24000)
seconds = len(audio) / 24000
print(f'{seconds:.1f} s, 24 kHz mono')
if seconds < 15:
    print('Short. 20-30 s gives a better voice; consider recording again.')

## 4. What you said

If you read the tester's passage **word for word**, leave this as it is.
If you said something else, set `USE_WHISPER = True` and it will be transcribed for you.

In [ ]:
TARGET_TEXT = "Good morning, everyone. Today we're going to look at how living things get their energy. Think about the last thing you ate. Where did that energy come from before it reached your plate? Every food chain starts with the sun. Plants catch its light, store it as sugar, and pass it on to the animals that eat them. By the end of this lesson, you'll be able to draw a simple food chain and explain each link. Ready? Let's begin."
USE_WHISPER = False

os.makedirs('/tmp/kvoicewalk/texts', exist_ok=True)
with open('/tmp/kvoicewalk/texts/dev.txt', 'w') as f:
    f.write(TARGET_TEXT)
print('Saved.')

## 5. Build the voice (the long step, a few hours)

It prints in three parts:
1. **Stock voices** rated against your recording (a few minutes).
2. **Interpolation search**: a long list of voice pairs such as `0 1 -1.50 af_heart ...`. Slow, and **saves nothing yet**.
3. **The walk**: `Step:` lines each time it gets closer, and `Progress: step N/10000` every 100 steps. `best_target_sim` is how close it sounds to you (0 to 1; the KVoiceWalk author got 0.93 after 10,000 steps).

It stops itself after 10 hours (Kaggle allows 12), so step 6 always gets to run.

In [ ]:
STEPS = 10000
%cd /tmp/kvoicewalk
whisper = '--transcribe_start' if USE_WHISPER else ''
!timeout 10h .venv/bin/python main.py --target_text ./texts/dev.txt --target_audio ./in/dev.wav {whisper} --interpolate_start --step_limit {STEPS} --output_name dev --device cuda --log_interval 100
%cd /kaggle/working

## 6. Save the best result as `dev.bin`
It lands in the notebook's Output (`/kaggle/working`), with `dev-preview.wav` to listen to.

In [ ]:
import re, shutil
from IPython.display import Audio, FileLink, display

pts = glob.glob('/tmp/kvoicewalk/out/dev_dev_*/dev_*.pt')
assert pts, ('No voice saved yet. Step 5 saves its first file at its first "Step:" line, '
             'after the interpolation search has finished.')
step = lambda p: int(re.match(r'dev_(\d+)_', os.path.basename(p)).group(1))
latest_run = max(os.path.dirname(p) for p in pts)
best = max((p for p in pts if os.path.dirname(p) == latest_run), key=step)
print('Best:', os.path.basename(best))

voice = torch.load(best, map_location='cpu', weights_only=True)
print('shape', tuple(voice.shape))
assert voice.numel() == 510 * 256, 'Unexpected voice shape'
voice.float().contiguous().numpy().astype('<f4').tofile('/kaggle/working/dev.bin')
assert os.path.getsize('/kaggle/working/dev.bin') == 522240
shutil.copy(best[:-3] + '.wav', '/kaggle/working/dev-preview.wav')
display(Audio('/kaggle/working/dev-preview.wav'))
display(FileLink('dev.bin'))

## 7. Use it in the tester

Download `dev.bin` (the link above, or Output → dev.bin), put it at `spikes/kokoro-tester/voices/dev.bin` and reload the tester: a **Load developer voice** button appears in section 2.